In [ ]:
import re
import time
import random
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

# Quero pegar as edições do pato donald da Abril
START_EDICAO = "http://www.guiadosquadrinhos.com/edicao/pato-donald-n-0/pa19211/144139"

BASE = "http://www.guiadosquadrinhos.com"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AcademicCrawler/1.0; +contact: virginio.ruan@academico.ifpb.edu.br)"
}

MESES_PT = {
    "janeiro": 1, "fevereiro": 2, "março": 3, "abril": 4, "maio": 5, "junho": 6,
    "julho": 7, "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12
}

# Como eu não quero derrubar o guia dos quadrinhos, eu vou deixar um sleep aleatório considerável
def sleep_polite(a=2.5, b=6.0):
    time.sleep(random.uniform(a, b))

def get_soup(html):
    return BeautifulSoup(html, "html.parser")

def fetch(url, session):
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def get_text_after_label(soup, label):
    strong = soup.find("strong", string=re.compile(rf"^{re.escape(label)}\s*$", re.I))
    if not strong:
        return None
    node = strong.next_sibling
    while node and isinstance(node, str) and node.strip() == "":
        node = node.next_sibling
    if not node:
        return None
    return node.get_text(" ", strip=True) if hasattr(node, "get_text") else str(node).strip()

# No site, a edição vem tipo "maio de 2000". Quero trazer (5,2000)
def parse_mes_ano(texto):
    if not texto:
        return (None, None)
    m = re.search(r"([A-Za-zçãõéíóúâêôà]+)\s+de\s+(\d{4})", texto.strip(), re.I)
    if not m:
        return (None, None)
    mes = MESES_PT.get(m.group(1).lower())
    ano = int(m.group(2))
    return (mes, ano)

def parse_issue_number(soup):
    h1 = soup.find("h1")
    if not h1:
        return None
    txt = h1.get_text(" ", strip=True)
    m = re.search(r"n[º°]\s*(\d+)", txt)
    return int(m.group(1)) if m else None

# Encontrar link da próxima edição
NEXT_TEXTS = {">", "›", "»", ">>", "próxima", "proxima", "próximo", "proximo"}
EDICAO_LINK_RE = re.compile(r"/edicao/[^\"'\s]+/ptd0031/\d+$", re.I)

def find_next_edicao_url(soup, current_url):
    # 1) heurística por texto do link (>, Próxima, etc.)
    for a in soup.find_all("a", href=True):
        txt = a.get_text(" ", strip=True).lower()
        if txt in NEXT_TEXTS or any(t in txt for t in ["próxima", "proxima"]):
            href = a["href"].strip()
            if href.startswith("javascript:"):
                continue
            full = urljoin(current_url, href)
            if EDICAO_LINK_RE.search(full):
                return full

    # fallback com salto: tenta cur_num+1, cur_num+2, ... até cur_num+MAX_SKIP
    cur_num = parse_issue_number(soup)
    if cur_num is None:
        return None

    MAX_SKIP = 30 
    hrefs = [
        a["href"].strip()
        for a in soup.find_all("a", href=True)
        if not a["href"].strip().startswith("javascript:")
    ]

    for k in range(1, MAX_SKIP + 1):
        target = f"pato-donald-o-n-{cur_num + k}"
        for href in hrefs:
            if target in href and "/ptd0031/" in href:
                return urljoin(current_url, href)

    return None

    # return None

def parse_issue_page(html, url):
    soup = get_soup(html)
    numero = parse_issue_number(soup)
    publicado = get_text_after_label(soup, "Publicado em:")
    mes, ano = parse_mes_ano(publicado)
    preco = get_text_after_label(soup, "Preço de capa:")
    return {
        "edicao_numero": numero,
        "mes": mes,
        "ano": ano,
        "preco_capa": preco,
        "url": url
    }, soup

# Loop principal: crawl recursivo seguindo “próxima edição”
def crawl_follow_next(out_csv="edicoes.csv", max_edicoes=2000):
    session = requests.Session()
    url = START_EDICAO

    rows = []
    seen_urls = set()

    for i in range(max_edicoes):
        if url in seen_urls:
            break
        seen_urls.add(url)

        html = fetch(url, session)
        data, soup = parse_issue_page(html, url)

        if data["edicao_numero"] is not None:
            rows.append(data)
            print(f"✓ edição {data['edicao_numero']}")

        if len(rows) % 50 == 0:
            pd.DataFrame(rows).drop_duplicates("edicao_numero").sort_values("edicao_numero") \
              .to_csv(out_csv, index=False, encoding="utf-8-sig")
            print("[checkpoint] salvei")

        next_url = find_next_edicao_url(soup, url)
        if not next_url:
            print("Não achei link de próxima edição. Parando.")
            break

        url = next_url
        sleep_polite()

    df = pd.DataFrame(rows).drop_duplicates("edicao_numero").sort_values("edicao_numero")
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[fim] Salvei {len(df)} edições em {out_csv}")

crawl_follow_next("edicoes.csv")